In [10]:
from dotenv import load_dotenv
load_dotenv()

from typing import Annotated
from typing_extensions import TypedDict
from langchain.chat_models import  init_chat_model
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

In [11]:
from langchain.chat_models import init_chat_model



In [25]:
llm = init_chat_model("google_genai:gemini-3.1-flash-lite")

class State(TypedDict):
    messages: Annotated[list, add_messages]

def chatbot(state: State) -> State:
    return {"messages":[llm.invoke(state["messages"])]}

builder = StateGraph(State)
builder.add_node("chatbot_node", chatbot)

builder.add_edge(START, "chatbot_node")
builder.add_edge("chatbot_node", END)

graph = builder.compile()

In [24]:
message = {"role":"user", "content": "Who is the God of Cricket ? Printonly name"}
response = graph.invoke({"messages": [message]})
response["messages"]

[HumanMessage(content='Who is the God of Cricket ? Printonly name', additional_kwargs={}, response_metadata={}, id='d18968c3-80a9-4672-93c8-7847bab174ed'),
 AIMessage(content=[{'type': 'text', 'text': 'Sachin Tendulkar', 'extras': {'signature': 'EjQKMgEMOdbHszH0d9aOlDQ+GdEjX6vy8vcJvz6cwpakEaR36cdTugCxGbu2+QBYis4sNa7V'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e610a-9ab3-76b0-9e2c-31ce0ee3fcab-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 11, 'output_tokens': 3, 'total_tokens': 14, 'input_token_details': {'cache_read': 0}})]

In [22]:
state = None

while True:
    in_message = input("You: ")
    if in_message.lower() in {"quit", "exit"}:
        break
    if state is None:
        state: State = {"messages": [{"role":"user", "content": in_message}]}
    else:
        state["messages"].append({"role":"user", "content": in_message})

    state = graph.invoke(state)
    print("Bot:", state["messages"][-1].content[0]["text"])



ChatGoogleGenerativeAIError: Error calling model 'gemini-3.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.5-flash\nPlease retry in 40.633975388s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeT